# 03 — Data Preprocessing & Cleaning
### AI Interview System — 100% Project-Owned ML Pipeline
This notebook executes deterministic cleaning on the dataset:
1. **Missing Value Detection & Reporting**: Explicitly tallies NULL, empty, and whitespace-only fields.
2. **Deterministic Imputation & Pruning**: Drops records with missing target questions; normalizes text casing and whitespace.
3. **Exact Duplicate Removal**: Deduplicates questions across domains while retaining original samples.
4. **Exporting Preprocessed Artifacts**: Saves clean intermediate data to `dataset/preprocessed/preprocessed_dataset.json`.


In [ ]:
# Cell 1: Environment Setup & Project Workspace Auto-Detection
import os
import sys
import json
import re
from pathlib import Path
from datetime import datetime, timezone

# Auto-detect workspace root (supports Google Colab, local terminal, or notebooks/ subfolder)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    if Path('/content/ai-interview-system/ml-service').exists():
        WORKSPACE_DIR = Path('/content/ai-interview-system/ml-service')
    elif Path('/content/drive/MyDrive/ai-interview-system/ml-service').exists():
        WORKSPACE_DIR = Path('/content/drive/MyDrive/ai-interview-system/ml-service')
    else:
        WORKSPACE_DIR = Path(os.getcwd())
    print("[OK] Running in Google Colab:", WORKSPACE_DIR)
except ImportError:
    cwd = Path(os.getcwd())
    if cwd.name == "notebooks":
        WORKSPACE_DIR = cwd.parent
    elif (cwd / "ml-service").exists():
        WORKSPACE_DIR = cwd / "ml-service"
    else:
        WORKSPACE_DIR = cwd
    print("[OK] Running in local environment:", WORKSPACE_DIR)

WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE_DIR)
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))
print(f"[OK] Working Directory set to: {WORKSPACE_DIR}")

RAW_DATASET_FILE = WORKSPACE_DIR / "dataset" / "raw" / "raw_interview_dataset.json"
PREPROCESSED_DIR = WORKSPACE_DIR / "dataset" / "preprocessed"
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_FILE = PREPROCESSED_DIR / "preprocessed_dataset.json"

with open(RAW_DATASET_FILE, "r", encoding="utf-8") as f:
    raw_records = json.load(f)

print(f"Loaded {len(raw_records)} raw records.")


In [ ]:
# Cell 2: Missing Value Detection & Cleaning
missing_counts = {"question": 0, "domain": 0, "difficulty": 0, "answer": 0}
cleaned_records = []
seen_questions = set()
duplicates_removed = 0

for r in raw_records:
    q = r.get("question", "")
    d = r.get("domain", "")
    diff = r.get("difficulty", "")
    a = r.get("answer", "")

    if not q or not q.strip():
        missing_counts["question"] += 1
        continue
    if not d or not d.strip():
        missing_counts["domain"] += 1
        d = "General Software Engineering"
    if not diff or not diff.strip():
        missing_counts["difficulty"] += 1
        diff = "Intermediate"
    if not a or not a.strip():
        missing_counts["answer"] += 1
        a = "Comprehensive technical explanation."

    # Normalize whitespace
    q_clean = re.sub(r"\s+", " ", q).strip()
    a_clean = re.sub(r"\s+", " ", a).strip()
    q_lower = q_clean.lower()

    if q_lower in seen_questions:
        duplicates_removed += 1
        continue
    seen_questions.add(q_lower)

    cleaned_records.append({
        "id": r.get("id", f"REC_{len(cleaned_records):04d}"),
        "domain": d.strip(),
        "difficulty": diff.strip(),
        "question": q_clean,
        "answer": a_clean
    })

print("Missing Values Detected:", missing_counts)
print(f"Exact Duplicates Removed: {duplicates_removed}")
print(f"Cleaned Records Remaining: {len(cleaned_records)}")

with open(PREPROCESSED_FILE, "w", encoding="utf-8") as f:
    json.dump(cleaned_records, f, indent=2)

print(f"Saved preprocessed dataset to: {PREPROCESSED_FILE}")
print("Stage 03 Completed Successfully.")
